## Pydantic Validators

- **`field_validator`** → validates a **specific field**.
- **`model_validator`** → validates the **whole model**, especially when multiple fields are involved.

```python
@field_validator("email")
@classmethod
def validate_email(cls, value):
    return value
```

```python
@model_validator(mode="after")
def validate_patient(self):
    if self.age < 18 and self.weight > 100:
        raise ValueError("Invalid patient data")
    return self
```
before → validate/modify raw input before Pydantic validation

after → validate the completed Pydantic model ✅

### 🧠 Remember

**One field → `field_validator`**  
**Multiple fields → `model_validator`**

Field Validator

In [3]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field,field_validator
from typing import List,Dict,Optional,Annotated

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        valid_domains = ['gmail.com', 'yahoo.com', 'outlook.com']
        domain = value.split('@')[-1]
        if domain not in valid_domains:
            raise ValueError(f"Email domain must be one of {valid_domains}")
        return value


# Create patient
patient = Patient(
    name="John",
    email="john@gmail.com",
    age=30,
    weight=70,
    allergies=[],
    contact_details={}
)

# Display patient
print(patient)

# Display as dictionary
print(patient.model_dump())

name='John' email='john@gmail.com' age=30 weight=70.0 allergies=[] contact_details={}
{'name': 'John', 'email': 'john@gmail.com', 'age': 30, 'weight': 70.0, 'allergies': [], 'contact_details': {}}


Model Validator

In [4]:
from pydantic import BaseModel, EmailStr, model_validator
from typing import List, Dict


class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode="after")
    def validate_patient(self):
        valid_domains = ["gmail.com", "yahoo.com", "outlook.com"]

        domain = self.email.split("@")[-1]

        if domain not in valid_domains:
            raise ValueError(
                f"Email domain must be one of {valid_domains}"
            )

        return self


# Create patient
patient = Patient(
    name="John",
    email="john@gmail.com",
    age=30,
    weight=70,
    allergies=[],
    contact_details={}
)

print(patient)
print(patient.model_dump())

name='John' email='john@gmail.com' age=30 weight=70.0 allergies=[] contact_details={}
{'name': 'John', 'email': 'john@gmail.com', 'age': 30, 'weight': 70.0, 'allergies': [], 'contact_details': {}}


## Computed Fields

**Validator** → Check/validate input ❗

**Computed field** → Calculate a value 🧮

Computed fields are useful when you want a **calculated value to behave like a field of your model**, including appearing in `model_dump()`.

```python
@computed_field
@property
def age_group(self):
    return "Adult" if self.age >= 18 else "Minor"
```

In [6]:
from pydantic import BaseModel, computed_field


class Patient(BaseModel):
    age: int

    @computed_field
    @property
    def age_group(self) -> str:
        return "Adult" if self.age >= 18 else "Minor"


# Create patient
patient = Patient(age=25)

# Access computed field
print(patient.age_group)

# Show complete model
print(patient)

# Show as dictionary
print(patient.model_dump())

Adult
age=25 age_group='Adult'
{'age': 25, 'age_group': 'Adult'}


## Pydantic Serialization & Nested Models

**Serialization**  
→ Pydantic object → `dict` / JSON

**Nested Model**  
→ One Pydantic model inside another Pydantic model

**`model_dump()`**  
→ Pydantic object → Python dictionary

**`model_dump_json()`**  
→ Pydantic object → JSON string